# Random Nucleotide Sampling

![Random Nucleotide Sampling](https://proto-bio.github.io/proto-assets/images/tool/random_nucleotide/hero.png)

This notebook demonstrates `run_random_nucleotide_sample`, which generates nucleotide sequence variants by filling masked positions with bases drawn from an IUPAC substitution scheme. It covers pre-masked sequences, automatic position selection via masking strategies, IUPAC scheme comparisons, RNA auto-detection, and batch library generation.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("random_nucleotide")
display_overview("random_nucleotide")
display_docs_section("random_nucleotide", "Background")

# Random Nucleotide Sampling

Random Nucleotide Sampling fills the masked positions of a DNA or RNA sequence with random bases drawn from an IUPAC ambiguity-code pool. Positions can be marked directly with `_` or selected automatically by a masking strategy, and the substitution alphabet is set by a single IUPAC code: `N` for any base, `R` for purines, `S` for strong pairs, and so on. It runs on CPU with no model and no external dependencies, and serves as an unbiased random baseline for sequence-design and library-generation workflows.

Random Nucleotide Sampling performs random mutagenesis at the nucleotide level: it takes a DNA or RNA sequence, determines which positions are designable, and replaces each with a base drawn uniformly from a chosen IUPAC degenerate-base pool. It generates nucleotide diversity without any learned model, the simplest possible baseline against which model-guided generators can be compared.

Internally, designable positions are either the `_` characters already present in the input or, when none are present, positions chosen by the configured masking strategy. Each masked position is filled independently by drawing one base uniformly at random from the pool that the IUPAC code expands to: `N` expands to A/C/G/T, `R` to A/G, `S` to G/C, and so on. Sampling is uniform within the pool, with no frequency weighting. When the input is RNA, sampled `T` bases are converted to `U`. With a fixed seed the output is deterministic.

This tool is original proto-tools code maintained by [Proto](https://github.com/evo-design/proto-tools).

## Available tools

In [2]:
display_available_tools("random_nucleotide")

- **`run_random_nucleotide_sample()`** — Sample nucleotide sequences by filling masked positions with random bases from an IUPAC substitution scheme

### `run_random_nucleotide_sample`

`run_random_nucleotide_sample` fills masked positions in DNA or RNA sequences with bases drawn from an IUPAC substitution scheme. Positions marked with `_` are replaced by a randomly sampled base; all other positions are preserved unchanged. The masking strategy system lets you automatically select which positions to diversify — by count, fraction, or specific indices — or you can pre-mask positions manually. The tool auto-detects DNA versus RNA by the presence of uracil and converts sampled thymine bases to uracil for RNA inputs.

In [3]:
from proto_tools import RandomNucleotideSampleInput, RandomNucleotideSampleConfig, run_random_nucleotide_sample
from proto_tools.transforms.masking import MaskingStrategy

In [4]:
# Display input docs
display_api_reference("random_nucleotide", "input", "run_random_nucleotide_sample")

# Underscore characters mark positions for random substitution
inputs = RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"])

**Input** — `RandomNucleotideSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA/RNA sequence(s) to mutate. May contain '_' at positions to sample. |

In [5]:
# Display config docs
display_api_reference("random_nucleotide", "config", "run_random_nucleotide_sample")

# Use a masking strategy to automatically select 2 positions to mutate | see docs above for all fields
config = RandomNucleotideSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=2),
    seed=42,
)

**Config** — `RandomNucleotideSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>masking_strategy</code> | <code>RandomMaskingStrategy</code> | <code>RandomMaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None)</code> | Controls which positions to mask for sampling. Default: random 30%. |
| <code>substitution_scheme</code> | <code>Literal['N', 'R', 'Y', 'S', 'W', 'K', 'M', 'B', 'D', 'H', 'V']</code> | <code>'N'</code> | IUPAC code defining the nucleotide substitution pool. |
| <code>sequence_type</code> | <code>Literal['auto', 'dna', 'rna']</code> | <code>'auto'</code> | Sequence type: auto-detect, force DNA, or force RNA. |

In [6]:
# Run the sampling function
wild_type = "ACGTACGTAC"
result = run_random_nucleotide_sample(
    RandomNucleotideSampleInput(sequences=[wild_type]),
    config,
)

In [7]:
# Display output docs
display_api_reference("random_nucleotide", "output", "run_random_nucleotide_sample")

# Show the variant and which positions were mutated
variant = result.sequences[0]
diffs = [i for i, (a, b) in enumerate(zip(wild_type, variant)) if a != b]
print(f"Wild type:         {wild_type}")
print(f"Variant:           {variant}")
print(f"Mutated positions: {diffs}")

# Demonstrate pre-masked input
result_premask = run_random_nucleotide_sample(RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"]))
print(f"\nPre-masked input:  ACGT_CGT_A")
print(f"Variant:           {result_premask.sequences[0]}")

# Demonstrate IUPAC substitution schemes
print("\nIUPAC scheme comparison (all-masked sequence):")
for scheme in ["N", "R", "Y", "S", "W"]:
    scheme_config = RandomNucleotideSampleConfig(substitution_scheme=scheme, seed=42)
    scheme_result = run_random_nucleotide_sample(
        RandomNucleotideSampleInput(sequences=["____"]),
        scheme_config,
    )
    print(f"  {scheme}: {scheme_result.sequences[0]}")

# Demonstrate RNA auto-detection
rna_result = run_random_nucleotide_sample(RandomNucleotideSampleInput(sequences=["AUGC_CGU"]))
print(f"\nRNA input:    AUGC_CGU")
print(f"RNA variant:  {rna_result.sequences[0]}  (T converted to U)")

**Output** — `RandomNucleotideSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Nucleotide sequences with masked positions randomly filled |

Wild type:         ACGTACGTAC
Variant:           ACGAACGTAA
Mutated positions: [3, 9]

Pre-masked input:  ACGT_CGT_A
Variant:           ACGTACGTAA

IUPAC scheme comparison (all-masked sequence):
  N: AAGC
  R: AAGA
  Y: CCTC
  S: GGCG
  W: AATA

RNA input:    AUGC_CGU
RNA variant:  AUGCCCGU  (T converted to U)


## Export Results

Sampling results can be saved to FASTA, TXT, or JSON format for downstream analysis or synthesis ordering.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generate a batch of 5 variants from the same wild-type sequence
batch_inputs = RandomNucleotideSampleInput(sequences=["ACGTACGTAC"] * 5)
batch_config = RandomNucleotideSampleConfig(masking_strategy=MaskingStrategy(num_mutations=2))
batch_result = run_random_nucleotide_sample(batch_inputs, batch_config)

batch_result.export(name="random_nucleotide_variants", export_path=output_dir, file_format="fasta")
print(f"Exported sequences to {output_dir / 'random_nucleotide_variants.fasta'}")

Exported sequences to example_output/random_nucleotide_variants.fasta
